# Okike Academy — Institutional 7-Stage Auto-Sync Video Worker (All Courses)

Runs the full **7-Stage Institutional Pipeline** (Planner → Critic → Scene Director → HTML/CSS UI Mockups → Kokoro-82M & -16 LUFS → 1080p Non-Overlapping Remotion + 10-Step LatentSync → QA Gate) automatically for **every course** on `https://academy.okikeenterprises.com`.

In [ ]:
import os, subprocess, time

SERVER_URL = "https://academy.okikeenterprises.com"
os.environ["GEMINI_API_KEY"] = "PASTE_YOUR_GEMINI_API_KEY_HERE"
os.environ["OPENROUTER_API_KEY"] = "PASTE_YOUR_OPENROUTER_API_KEY_HERE"
# Direct-to-Cloudinary upload (bypasses Vercel's 4.5 MB serverless body limit for 1080p MP4s)
os.environ["CLOUDINARY_CLOUD_NAME"] = "PASTE_YOUR_CLOUDINARY_CLOUD_NAME"
os.environ["CLOUDINARY_API_KEY"] = "PASTE_YOUR_CLOUDINARY_API_KEY"
os.environ["CLOUDINARY_API_SECRET"] = "PASTE_YOUR_CLOUDINARY_API_SECRET"

# 1. Clone/update Okike Academy repo (includes worker/pipeline, worker/remotion_studio, and both mascots)
if not os.path.exists("okike-academy"):
    !git clone -q https://github.com/OKIKE-AFRICAN-LIMITED/okike-academy.git
else:
    !cd okike-academy && git pull -q

# 2. Install system & Python dependencies + Playwright Chromium for crisp HTML/CSS UI mockups
!apt-get -qq update && apt-get -qq install -y ffmpeg espeak-ng libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 > /dev/null
!pip -q install playwright kokoro-onnx soundfile openai-whisper diffusers transformers accelerate omegaconf einops opencv-python mediapipe insightface onnxruntime-gpu kornia scenedetect ffmpeg-python decord huggingface_hub DeepCache face_alignment av lpips requests
!playwright install chromium

# 3. LatentSync & Kokoro weights
from huggingface_hub import hf_hub_download
if not os.path.exists("okike-academy/worker/LatentSync"):
    !git clone -q https://github.com/bytedance/LatentSync.git okike-academy/worker/LatentSync
os.makedirs("okike-academy/worker/LatentSync/checkpoints/whisper", exist_ok=True)
hf_hub_download("ByteDance/LatentSync-1.5", "latentsync_unet.pt", local_dir="okike-academy/worker/LatentSync/checkpoints")
hf_hub_download("ByteDance/LatentSync-1.5", "whisper/tiny.pt", local_dir="okike-academy/worker/LatentSync/checkpoints")

!wget -q -nc -P okike-academy/worker https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx
!wget -q -nc -P okike-academy/worker https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin
!cp okike-academy/worker/mascots/host_1.mp4 okike-academy/worker/host_1.mp4
!cp okike-academy/worker/mascots/host_2.mp4 okike-academy/worker/host_2.mp4
!cd okike-academy/worker/remotion_studio && npm install --silent

# 4. Start Institutional Auto-Sync Worker (watches ALL courses on academy.okikeenterprises.com!)
subprocess.run(["pkill", "-f", "render_explainer_v5.py"], check=False)
log = open("okike-academy/worker/institutional_worker.log", "w")
p = subprocess.Popen(
    ["python", "-u", "render_explainer_v5.py", "--server-url", SERVER_URL, "--watch"],
    cwd="okike-academy/worker",
    stdout=log,
    stderr=subprocess.STDOUT,
)
time.sleep(4)
print("Okike Institutional 7-Stage Worker LIVE! PID:", p.pid, "| Alive:", p.poll() is None)
!tail -n 20 okike-academy/worker/institutional_worker.log